# 🚀 학생 건강상태 분류 — 최종 성능 개선 실험

현재 Balanced Accuracy는 약 `0.94864`이며, 목표는 `0.952` 이상이다.

지금 단계에서는 새로운 모델 하나를 추가하기보다, 앞에서 만든 앙상블을 기준으로 여러 가지 개선 방법을 하나씩 적용하고 효과를 비교한다. 각 실험은 OOF 점수와 리더보드 점수를 함께 확인해 실제로 도움이 되는지 판단

## 이번에 확인할 내용

| 실험 | 확인할 내용 |
|---|---|
| 원본 데이터 결합 | 대회의 원본 데이터가 제공되는지 확인하고, 사용할 수 있다면 성능 변화를 비교 |
| 스태킹 | 기존 OOF 예측을 메타모델의 입력으로 사용해 단순 평균보다 나은지 확인 |
| 하이퍼파라미터 튜닝 | Optuna를 사용해 부스팅 모델의 주요 파라미터를 조정 |
| Seed 평균 | 여러 random seed로 학습한 결과를 평균내 예측의 변동을 줄임 |
| Fold 수 증가 | 최종 학습에서 10-Fold를 사용했을 때 성능이 달라지는지 확인 |
| 클래스별 확률 보정 | Balanced Accuracy 기준에서 클래스별 예측 비율을 조정해 성능 변화를 확인 |

모든 방법을 한꺼번에 적용하지 않고, 한 가지씩 실험한 뒤 결과를 기록한다. 성능이 좋아지지 않거나 재현이 어려운 방법은 최종 모델에서 제외

`class_weight='balanced'`는 클래스별 데이터 수 차이를 보정하기 위해 계속 유지하되, 실제 OOF 성능과 클래스별 Recall을 함께 확인

## 실행 시간 줄이기

앙상블 실험을 반복할 때는 이미 학습한 모델의 OOF와 Test 예측 확률을 파일로 저장해 재사용한다. 이렇게 하면 가중치나 스태킹 조합을 바꿀 때 모델을 다시 학습하지 않아도 됨

GPU를 사용할 수 있는 환경에서는 XGBoost와 CatBoost의 GPU 옵션도 비교해 본다. 다만 GPU 사용 여부에 따라 결과가 달라지지 않도록 random seed와 평가 조건은 동일하게 유지

## 0. 설치 & 설정

**실행 전략 (3단계):**
1. `RUN_FRACTION=0.3, SEEDS=[42], RUN_OPTUNA=False` → 전체 흐름 검증 (빠름)
2. `RUN_FRACTION=1.0` → 베이스 앙상블 제출, 리더보드 확인
3. `RUN_OPTUNA=True, SEEDS=[42,2026,7], N_SPLITS=10` → 최종 풀파워 (GPU 필수)


In [1]:
!pip install -q lightgbm xgboost catboost optuna kagglehub
print('설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 8.2 MB/s eta 0:00:00
설치 완료


In [2]:
import time, warnings, os, json, hashlib
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# ═══════════ 핵심 설정 ═══════════
RUN_FRACTION     = 0.3      # ① 0.3으로 검증 → ② 최종 1.0
N_SPLITS         = 5        # 실험 5 → ③ 최종 제출 때 10
SEEDS            = [42]     # ③ 최종엔 [42, 2026, 7]  (seed averaging)
RUN_OPTUNA       = False    # ③ 튜닝 (GPU + 1~2시간 여유 있을 때 True)
OPTUNA_TRIALS    = 30
USE_CLASS_WEIGHT = True     # 지표 = balanced accuracy 확정 → True 고정!
ORIGINAL_SLUG    = ''       # ⭐ Data 탭에서 원본 데이터셋 확인 → 'owner/dataset-name'
# ═════════════════════════════════

RS = 42; np.random.seed(RS)
score_fn = balanced_accuracy_score
HAS_GPU = (os.system('nvidia-smi > /dev/null 2>&1') == 0)
print('GPU:', '사용 가능 ✅' if HAS_GPU else '❌ 없음 → 상단 메뉴 [런타임]-[런타임 유형 변경]-[T4 GPU] 강력 권장')

GPU: ❌ 없음 → 상단 메뉴 [런타임]-[런타임 유형 변경]-[T4 GPU] 강력 권장


## 1. 데이터 로드 + 파생변수 (Lv4와 동일, self-contained)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_PATH='/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'
train=pd.read_csv(DATA_PATH+'train.csv'); test=pd.read_csv(DATA_PATH+'test.csv')
TARGET,ID='health_condition','id'
numeric_features=['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
categorical_features=['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']
feature_cols=numeric_features+categorical_features
HR_HI=train['heart_rate'].quantile(0.75); STEP_LO=train['step_count'].quantile(0.25); WATER_LO=train['water_intake'].quantile(0.25)
ord_maps={'stress_level':{'low':0,'medium':1,'high':2},'sleep_quality':{'poor':0,'average':1,'good':2},
 'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},'smoking_alcohol':{'no':0,'occasional':1,'yes':2}}
# 명목형 코드 매핑 (train 기준 고정 → test/원본에도 동일 적용)
code_maps={c:{k:i for i,k in enumerate(train[c].astype('category').cat.categories)} for c in ['diet_type','gender']}

def make_features(df):
    X=df.copy()
    for col,m in ord_maps.items(): X[col+'_ord']=X[col].map(m)
    X['sleep_debt']=(7-X['sleep_duration']).clip(lower=0); X['sleep_excess']=(X['sleep_duration']-9).clip(lower=0)
    X['sleep_ideal']=X['sleep_duration'].between(7,9).astype('float'); X.loc[X['sleep_duration'].isna(),'sleep_ideal']=np.nan
    X['bmi_cat']=pd.cut(X['bmi'],bins=[-np.inf,18.5,25,30,np.inf],labels=[0,1,2,3]).astype('float')
    X['bmi_abnormal']=(~X['bmi'].between(18.5,25)).astype('float'); X.loc[X['bmi'].isna(),'bmi_abnormal']=np.nan
    X['hr_high']=(X['heart_rate']>HR_HI).astype('float'); X.loc[X['heart_rate'].isna(),'hr_high']=np.nan
    X['steps_per_ex_min']=X['step_count']/(X['exercise_duration']+1); X['cal_per_step']=X['calorie_expenditure']/(X['step_count']+1)
    X['low_activity']=(X['step_count']<STEP_LO).astype('float'); X.loc[X['step_count'].isna(),'low_activity']=np.nan
    X['low_water']=(X['water_intake']<WATER_LO).astype('float'); X.loc[X['water_intake'].isna(),'low_water']=np.nan
    risk=pd.DataFrame(index=X.index)
    risk['a']=(X['sleep_duration']<6).astype(float); risk['b']=(X['stress_level']=='high').astype(float)
    risk['c']=(X['sleep_quality']=='poor').astype(float); risk['d']=(X['physical_activity_level']=='sedentary').astype(float)
    risk['e']=(X['step_count']<STEP_LO).astype(float); risk['f']=(X['smoking_alcohol']=='yes').astype(float)
    risk['g']=(~X['bmi'].between(18.5,25)).astype(float)
    X['lifestyle_risk_score']=risk.sum(axis=1); X['n_missing']=df[feature_cols].isnull().sum(axis=1)
    for c in feature_cols: X[c+'_isna']=df[c].isnull().astype('int8')
    for c,mp in code_maps.items(): X[c+'_code']=X[c].map(mp)
    return X

train_fe=make_features(train); test_fe=make_features(test)
print('train_fe', train_fe.shape, '| test_fe', test_fe.shape)

train_fe (690088, 46) | test_fe (295753, 45)


In [5]:
# 변수 세트 — Lv4 진단(PRUNED vs FULL)에서 이긴 쪽으로 지정하세요. 기본 = FULL
FEATURES_PRUNED = numeric_features + [
    'lifestyle_risk_score','sleep_debt','sleep_ideal','bmi_cat','cal_per_step','steps_per_ex_min','low_activity',
    'stress_level_ord','sleep_quality_ord','physical_activity_level_ord','smoking_alcohol_ord',
    'diet_type_code','gender_code']
exclude=[ID,TARGET]+categorical_features
FEATURES_FULL=[c for c in train_fe.columns if c not in exclude and train_fe[c].dtype!='object']
BEST_FEATURES = FEATURES_FULL          # ← Lv4에서 PRUNED가 이겼다면 FEATURES_PRUNED로 변경
print('사용 변수:', len(BEST_FEATURES), '개')

le=LabelEncoder().fit(train_fe[TARGET]); CLASSES=le.classes_
if RUN_FRACTION<1.0:
    run=train_fe.groupby(TARGET,group_keys=False).apply(lambda s: s.sample(frac=RUN_FRACTION,random_state=RS))
else:
    run=train_fe
y_run=le.transform(run[TARGET])
print('실행 데이터:', run.shape, '| 클래스:', dict(zip(CLASSES, np.bincount(y_run))))

사용 변수: 38 개
실행 데이터: (207026, 46) | 클래스: {'at-risk': np.int64(177768), 'fit': np.int64(11941), 'unhealthy': np.int64(17317)}


## 2. ⭐ 원본 데이터 결합 실험

Playground 대회의 Train/Test 데이터가 특정 원본 데이터셋을 바탕으로 생성된 경우, 원본 데이터를 학습에 함께 사용했을 때 성능이 좋아지는지 확인할 수 있음

다만 원본 데이터가 항상 도움이 되는 것은 아니므로, 실제로 사용할 수 있는 데이터인지 먼저 확인하고 결합 전후의 OOF 성능을 비교

### 누수 방지 방법

원본 데이터를 모든 과정에 무조건 포함하지 않고, 각 Fold의 학습 데이터에만 추가한다.

- Fold의 학습 부분: 대회 데이터 + 원본 데이터
- Fold의 검증 부분: 대회 데이터만 사용
- OOF 점수: 대회 검증 데이터로만 계산

이렇게 해야 검증 데이터에 원본 정보가 섞이지 않고, 대회 데이터만으로 평가했을 때의 성능을 확인할 수 있다.

원본 데이터를 추가했을 때 OOF Balanced Accuracy와 클래스별 Recall이 함께 좋아지는 경우에만 최종 학습에 사용하는 방향으로 진행

In [6]:
orig_fe, y_orig = None, None
if ORIGINAL_SLUG:
    try:
        import kagglehub
        p = kagglehub.dataset_download(ORIGINAL_SLUG)
        csvs=[os.path.join(r,f) for r,_,fs in os.walk(p) for f in fs if f.endswith('.csv')]
        print('발견된 CSV:', [os.path.basename(c) for c in csvs])
        orig = pd.read_csv(csvs[0])
        print('원본 shape:', orig.shape, '\n컬럼:', list(orig.columns))

        # 컬럼명이 다르면 여기서 매핑하세요. 예: orig = orig.rename(columns={'Sleep Duration':'sleep_duration'})
        RENAME = {}
        orig = orig.rename(columns=RENAME)

        need = set(feature_cols+[TARGET])
        if need.issubset(orig.columns) and set(orig[TARGET].dropna().unique()).issubset(set(CLASSES)):
            orig = orig.drop_duplicates(subset=feature_cols)
            orig_fe = make_features(orig)
            y_orig = le.transform(orig_fe[TARGET])
            print(f'✅ 원본 {len(orig_fe)}행 결합 준비 완료 (fold 학습부분에만 추가됨)')
        else:
            print('⚠️ 컬럼/라벨 불일치 → RENAME 딕셔너리로 컬럼명을 맞춰주세요. 부족한 컬럼:', need-set(orig.columns))
    except Exception as e:
        print('원본 로드 실패 → 스킵:', str(e)[:120])
else:
    print('ORIGINAL_SLUG 미지정 → 원본 결합 스킵 (Data 탭에서 꼭 확인해 보세요! 가장 싼 +0.001~0.005 입니다)')

ORIGINAL_SLUG 미지정 → 원본 결합 스킵 (Data 탭에서 꼭 확인해 보세요! 가장 싼 +0.001~0.005 입니다)


## 3. OOF 엔진 v2 — 캐싱 + GPU + 원본증강 + 파라미터 주입

Lv4 엔진의 업그레이드판. 핵심 변화:

- **캐싱**: (모델, seed, fold수, fraction, 변수, 파라미터)가 같으면 저장된 확률을 즉시 로드 → **"Lv3처럼 오래 걸리는" 문제의 근본 해결.** 앙상블/스태킹/보정 실험은 재학습 없이 몇 초.
- **GPU**: XGBoost(`device='cuda'`)·CatBoost(`task_type='GPU'`)는 GPU에서 학습. (LightGBM·HistGB는 CPU가 안정적)
- **원본증강**: `orig_fe`가 있으면 각 fold의 **학습 부분에만** concat.
- **파라미터 주입**: Optuna가 찾은 best params를 그대로 넣을 수 있음.


In [7]:
CACHE = DATA_PATH + 'oof_cache/'; os.makedirs(CACHE, exist_ok=True)

def make_model(name, seed, params=None):
    p = params or {}
    if name=='LightGBM':
        from lightgbm import LGBMClassifier
        base=dict(n_estimators=3000, learning_rate=0.03, num_leaves=63, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, verbose=-1)
        base.update(p); return LGBMClassifier(random_state=seed, **base)
    if name=='XGBoost':
        from xgboost import XGBClassifier
        base=dict(n_estimators=3000, learning_rate=0.03, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                  early_stopping_rounds=100, eval_metric='mlogloss', n_jobs=-1, verbosity=0)
        if HAS_GPU: base['device']='cuda'
        base.update(p); return XGBClassifier(random_state=seed, **base)
    if name=='CatBoost':
        from catboost import CatBoostClassifier
        base=dict(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0, verbose=0)
        if HAS_GPU: base['task_type']='GPU'
        base.update(p); return CatBoostClassifier(random_state=seed, **base)
    if name=='HistGB':
        from sklearn.ensemble import HistGradientBoostingClassifier
        base=dict(max_iter=3000, learning_rate=0.03, max_leaf_nodes=63, l2_regularization=1.0,
                  early_stopping=True, validation_fraction=0.1, n_iter_no_change=100)
        base.update(p); return HistGradientBoostingClassifier(random_state=seed, **base)

def fit_one(name, model, Xtr, ytr, Xva, yva, sw):
    if name=='LightGBM':
        from lightgbm import early_stopping, log_evaluation
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=[(Xva,yva)],
                  callbacks=[early_stopping(100,verbose=False),log_evaluation(0)])
    elif name=='XGBoost':
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=[(Xva,yva)],verbose=False)
    elif name=='CatBoost':
        model.fit(Xtr,ytr,sample_weight=sw,eval_set=(Xva,yva),early_stopping_rounds=100,verbose=0)
    else:
        model.fit(Xtr,ytr,sample_weight=sw)
    return model

def run_oof(name, features, seed=42, params=None):
    key = hashlib.md5(json.dumps(dict(n=name,f=features,s=seed,k=N_SPLITS,fr=RUN_FRACTION,
          cw=USE_CLASS_WEIGHT,orig=orig_fe is not None,p=params), sort_keys=True, default=str
          ).encode()).hexdigest()[:12]
    fpath = CACHE + f'{name}_s{seed}_{key}.npz'
    if os.path.exists(fpath):                     # ← 캐시 히트: 재학습 없음!
        z=np.load(fpath); sc=score_fn(y_run, z['oof'].argmax(1))
        print(f'  {name:9s} seed{seed} 💾캐시  OOF={sc:.5f}'); return z['oof'], z['test'], sc
    X=run[features]; Xtest=test_fe[features]; n_cls=len(CLASSES)
    oof=np.zeros((len(X),n_cls)); test_proba=np.zeros((len(Xtest),n_cls))
    skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed); t0=time.time()
    for tr,va in skf.split(X,y_run):
        Xtr,ytr=X.iloc[tr],y_run[tr]; Xva,yva=X.iloc[va],y_run[va]
        if orig_fe is not None:                   # 원본은 학습부분에만 (검증은 대회 데이터만!)
            Xtr=pd.concat([Xtr,orig_fe[features]]); ytr=np.concatenate([ytr,y_orig])
        sw=compute_sample_weight('balanced',ytr) if USE_CLASS_WEIGHT else None
        m=fit_one(name,make_model(name,seed,params),Xtr,ytr,Xva,yva,sw)
        oof[va]=m.predict_proba(Xva); test_proba+=m.predict_proba(Xtest)/N_SPLITS
    sc=score_fn(y_run,oof.argmax(1))
    np.savez_compressed(fpath,oof=oof,test=test_proba)
    print(f'  {name:9s} seed{seed}  OOF={sc:.5f}  ({time.time()-t0:.0f}s, 캐시 저장)')
    return oof,test_proba,sc
print('OOF 엔진 v2 준비 완료 ✅')

OOF 엔진 v2 준비 완료 ✅


## 4. (선택) Optuna로 하이퍼파라미터 조정

Lv3에서는 RandomizedSearch를 15회 실행해 기본적인 파라미터 비교만 진행했다. 이번에는 Optuna를 사용해 이전 평가 결과를 참고하면서 다음 후보를 선택하는 방식으로 추가 튜닝을 진행

=> 모델의 성능을 높일 수 있는 설정을 효율적으로 찾기 위해 거칠 수 있는 단계

RandomizedSearch는 정해진 범위에서 후보를 무작위로 선택합니다. 반면 Optuna는 이전 실험 결과를 참고해 성능이 좋을 가능성이 높은 조합을 다음 후보로 선택

In [8]:
PARAM_FILE = DATA_PATH + 'best_params.json'
BEST_PARAMS = json.load(open(PARAM_FILE)) if os.path.exists(PARAM_FILE) else {}

if RUN_OPTUNA:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    Xh, Xv, yh, yv = train_test_split(run[BEST_FEATURES], y_run, test_size=0.2,
                                      stratify=y_run, random_state=RS)
    if orig_fe is not None:
        Xh=pd.concat([Xh,orig_fe[BEST_FEATURES]]); yh=np.concatenate([yh,y_orig])
    swh=compute_sample_weight('balanced',yh) if USE_CLASS_WEIGHT else None

    def space(name,t):
        if name=='LightGBM': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            num_leaves=t.suggest_int('num_leaves',31,255),
            min_child_samples=t.suggest_int('min_child_samples',10,100),
            subsample=t.suggest_float('subsample',0.6,1.0),
            colsample_bytree=t.suggest_float('colsample_bytree',0.5,1.0),
            reg_lambda=t.suggest_float('reg_lambda',0.01,10,log=True))
        if name=='XGBoost': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            max_depth=t.suggest_int('max_depth',4,10),
            min_child_weight=t.suggest_int('min_child_weight',1,20),
            subsample=t.suggest_float('subsample',0.6,1.0),
            colsample_bytree=t.suggest_float('colsample_bytree',0.5,1.0),
            reg_lambda=t.suggest_float('reg_lambda',0.01,10,log=True))
        if name=='CatBoost': return dict(
            learning_rate=t.suggest_float('learning_rate',0.01,0.1,log=True),
            depth=t.suggest_int('depth',4,9),
            l2_leaf_reg=t.suggest_float('l2_leaf_reg',0.5,20,log=True))

    for name in ['LightGBM','XGBoost','CatBoost']:
        def objective(t):
            m=fit_one(name,make_model(name,RS,space(name,t)),Xh,yh,Xv,yv,swh)
            return score_fn(yv,m.predict_proba(Xv).argmax(1))
        st=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=RS))
        t0=time.time(); st.optimize(objective,n_trials=OPTUNA_TRIALS,show_progress_bar=True)
        BEST_PARAMS[name]=st.best_params
        print(f'{name}: holdout={st.best_value:.5f} ({time.time()-t0:.0f}s)\n  {st.best_params}')
    json.dump(BEST_PARAMS,open(PARAM_FILE,'w'),indent=1)
    print('저장:',PARAM_FILE)
else:
    print('RUN_OPTUNA=False | 저장된 파라미터:', list(BEST_PARAMS.keys()) or '없음(기본값 사용)')

RUN_OPTUNA=False | 저장된 파라미터: 없음(기본값 사용)


## 5. 멀티시드 × 4모델 OOF 학습

같은 모델이라도 random seed가 달라지면 Fold 분할과 학습 과정이 달라져 예측 결과가 조금씩 달라질 수 있음

이번에는 4개 모델을 여러 seed로 학습한 뒤 OOF와 Test 예측 확률을 평균내어, 특정 seed에 따른 결과 차이를 줄이는지 확인한다.

- 사용 모델: 4개 부스팅 모델
- seed 수: 3개
- 총 학습 조합: 4 × 3 = 12개

이미 학습한 조합의 결과는 캐시에 저장한다.
-> 실행이 중간에 멈추더라도 다시 실행하면 완료된 조합은 건너뛰고 남은 조합부터 진행

In [9]:
MODELS=['LightGBM','XGBoost','CatBoost','HistGB']
runs={}   # (model, seed) -> (oof, test, score)
print(f'▶ {len(MODELS)}모델 × {len(SEEDS)}seed OOF 학습')
for name in MODELS:
    for seed in SEEDS:
        try: runs[(name,seed)]=run_oof(name,BEST_FEATURES,seed,BEST_PARAMS.get(name))
        except Exception as e: print(f'  {name} seed{seed} 실패: {str(e)[:80]}')

tbl=pd.DataFrame([{'model':n,'seed':s,'OOF_bal_acc':sc} for (n,s),(o,t,sc) in runs.items()])
display(tbl.sort_values('OOF_bal_acc',ascending=False).round(5))

▶ 4모델 × 1seed OOF 학습
  LightGBM  seed42 💾캐시  OOF=0.92214
  XGBoost   seed42 💾캐시  OOF=0.92664
  CatBoost  seed42 💾캐시  OOF=0.94824
  HistGB    seed42 💾캐시  OOF=0.94800


,model,seed,OOF_bal_acc
2,CatBoost,42,0.94824
3,HistGB,42,0.94800
1,XGBoost,42,0.92664
0,LightGBM,42,0.92214


## 6. ⭐ 앙상블 방식 비교

이번에는 같은 OOF 예측을 사용해 세 가지 앙상블 방식을 비교한다.

- **단순 평균**: 4개 모델의 예측 확률을 동일한 비율로 평균낸다.
- **가중 평균**: 모델별 성능을 참고해 서로 다른 가중치를 적용한다.
- **스태킹**: 4개 모델의 OOF 확률을 입력값으로 사용해 로지스틱 회귀 메타모델을 학습한다.

단순 평균과 가중 평균은 사람이 정한 비율에 따라 결과가 결정된다. 스태킹은 모델별 예측 조합을 학습하기 때문에, 특정 모델의 예측을 다른 모델의 결과와 함께 해석할 수 있다.

스태킹 성능은 `cross_val_predict`로 별도 교차검증해 평가한다. 이미 OOF로 만든 예측값을 다시 학습과 평가에 나누어 사용하여, 메타모델 평가 과정에서도 데이터 누수가 발생하지 않도록 한다.

세 방식의 Balanced Accuracy와 클래스별 Recall을 비교한 뒤, 실제 성능이 가장 안정적인 방식을 선택한다.

In [10]:
keys=list(runs.keys())
oof_stack=np.stack([runs[k][0] for k in keys])    # (n_runs, n_samples, 3)
test_stack=np.stack([runs[k][1] for k in keys])

# ① 단순 평균
oof_mean=oof_stack.mean(0); test_mean=test_stack.mean(0)
s_mean=score_fn(y_run,oof_mean.argmax(1))

# ② 가중 평균 (Dirichlet 랜덤서치 4000회)
rng=np.random.RandomState(RS); best_w=np.ones(len(keys))/len(keys); s_w=s_mean
for _ in range(4000):
    w=rng.dirichlet(np.ones(len(keys)))
    s=score_fn(y_run,np.tensordot(w,oof_stack,axes=(0,0)).argmax(1))
    if s>s_w: s_w,best_w=s,w
oof_wavg=np.tensordot(best_w,oof_stack,axes=(0,0)); test_wavg=np.tensordot(best_w,test_stack,axes=(0,0))

# ③ 스태킹 (메타 = 로지스틱 회귀)
meta_X=np.hstack([runs[k][0] for k in keys])      # (n_samples, n_runs*3)
meta_T=np.hstack([runs[k][1] for k in keys])
meta=LogisticRegression(max_iter=2000,class_weight='balanced' if USE_CLASS_WEIGHT else None)
oof_meta_pred=cross_val_predict(meta,meta_X,y_run,cv=StratifiedKFold(5,shuffle=True,random_state=RS),
                                method='predict_proba',n_jobs=-1)
s_stack=score_fn(y_run,oof_meta_pred.argmax(1))
meta.fit(meta_X,y_run); test_stackp=meta.predict_proba(meta_T)

cmp=pd.DataFrame({'전략':['① 단순평균','② 가중평균','③ 스태킹'],'OOF_bal_acc':[s_mean,s_w,s_stack]})
display(cmp.round(5))
best_i=int(np.argmax([s_mean,s_w,s_stack]))
FINAL_OOF=[oof_mean,oof_wavg,oof_meta_pred][best_i]
FINAL_TEST=[test_mean,test_wavg,test_stackp][best_i]
print(f'채택: {cmp["전략"][best_i]}  OOF={cmp["OOF_bal_acc"][best_i]:.5f}')
print('(참고: ②는 OOF에 과적합 여지가 있어, ①과 차이가 +0.0005 미만이면 ①이나 ③을 믿는 게 안전)')

,전략,OOF_bal_acc
0,① 단순평균,0.94262
1,② 가중평균,0.94867
2,③ 스태킹,0.94817


채택: ② 가중평균  OOF=0.94867
(참고: ②는 OOF에 과적합 여지가 있어, ①과 차이가 +0.0005 미만이면 ①이나 ③을 믿는 게 안전)


## 7. Balanced Accuracy를 높이기 위한 마지막 튜닝

이번 프로젝트의 목표 평가지표는 **Balanced Accuracy**였다. 이 지표는 `fit`, `at_risk`, `unhealthy` 세 클래스의 Recall을 동일한 비중으로 평균 내기 때문에, 특정 클래스만 잘 맞추는 모델보다 **세 클래스를 균형 있게 예측하는 모델**이 더 높은 점수를 얻는다.

모델을 학습하면서 OOF(Out-Of-Fold) 예측 결과를 확인해 보니, `at_risk`는 비교적 안정적으로 예측했지만 `fit`과 `unhealthy`는 경계에 있는 샘플들이 서로 헷갈리는 경우가 종종 있었다. 특히 예측 확률 차이가 크지 않은 샘플은 `argmax`에 의해 다른 클래스로 분류되면서 해당 클래스의 Recall이 낮아지는 모습을 확인했다.

그래서 마지막 단계에서는 **클래스별 예측 확률을 보정하는 후처리(Post-processing)** 를 적용했다. OOF 데이터에서 `fit`과 `unhealthy` 클래스의 예측 확률에 여러 배수(multiplier)를 적용해 보며 Balanced Accuracy가 가장 높게 나오는 값을 찾았고, 검증 과정에서 찾은 동일한 배수를 Test 데이터에도 그대로 적용했다.

이 방법은 모델을 다시 학습시키는 것이 아니라 **최종 예측 단계에서 클래스 간의 결정 경계를 미세하게 조정하는 과정**이다. 큰 변화는 아니었지만, Balanced Accuracy처럼 각 클래스의 Recall을 중요하게 평가하는 환경에서는 이러한 작은 튜닝도 성능 향상에 도움이 된다는 점을 경험할 수 있었음


In [11]:
def tune_multipliers(proba,y,iters=5000):
    rng=np.random.RandomState(RS); best=np.ones(proba.shape[1])
    best_s=score_fn(y,proba.argmax(1))
    for _ in range(iters):
        m=np.exp(rng.normal(0,0.15,proba.shape[1]))     # 1.0 근처 랜덤 배수
        s=score_fn(y,(proba*m).argmax(1))
        if s>best_s: best_s,best=s,m
    return best,best_s

base_s=score_fn(y_run,FINAL_OOF.argmax(1))
mult,cal_s=tune_multipliers(FINAL_OOF,y_run)
print(f'보정 전 OOF={base_s:.5f} → 보정 후 OOF={cal_s:.5f} (+{cal_s-base_s:.5f})')
print('클래스 배수:', dict(zip(CLASSES,mult.round(3))))
USE_CAL = (cal_s-base_s) > 0.0003          # 이득이 노이즈 수준이면 미적용 (OOF 과적합 방지)
print('적용 여부:', USE_CAL)

보정 전 OOF=0.94867 → 보정 후 OOF=0.94934 (+0.00067)
클래스 배수: {'at-risk': np.float64(0.65), 'fit': np.float64(0.996), 'unhealthy': np.float64(1.305)}
적용 여부: True


## 8. 제출 파일 생성


In [12]:
final_proba = FINAL_TEST*mult if USE_CAL else FINAL_TEST
pred=le.inverse_transform(final_proba.argmax(1))
tag=('full' if RUN_FRACTION>=1.0 else f'frac{RUN_FRACTION}')+f'_{len(SEEDS)}seed_{N_SPLITS}fold'
path=DATA_PATH+f'submission_lv5_{tag}.csv'
pd.DataFrame({ID:test[ID],TARGET:pred}).to_csv(path,index=False)
print('저장:',path)
print('예측 분포:',pd.Series(pred).value_counts(normalize=True).round(3).to_dict())
print(f'\n기대 리더보드 ≈ OOF {score_fn(y_run,(FINAL_OOF*mult if USE_CAL else FINAL_OOF).argmax(1)):.5f}')

저장: /content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/submission_lv5_frac0.3_1seed_5fold.csv
예측 분포: {'at-risk': 0.807, 'unhealthy': 0.12, 'fit': 0.073}

기대 리더보드 ≈ OOF 0.94934
